# Labels, Costs, and Infos

This notebook slows down the MASA environment loop. We will step through `colour_grid_world` by hand and inspect observations, rewards, labels, costs, constraint metrics, termination, and truncation.

The matching static docs page is at `docs/Tutorials/Basics/Labels Costs and Infos.md`.

## CPU-first setup

Set these before importing MASA/JAX modules. This tutorial does not train an agent, but the same CPU-first convention keeps all tutorials portable.

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

from pprint import pprint

ACTION_NAMES = {
    0: "left",
    1: "right",
    2: "down",
    3: "up",
    4: "stay",
}


## Inspect labels and costs directly

MASA separates environment observations from semantic safety signals. A `label_fn` maps an observation to atomic propositions, and a `cost_fn` maps labels to a scalar safety cost.

In [ ]:
from masa.envs.tabular.colour_grid_world import (
    BLUE_STATE,
    GOAL_STATE,
    START_STATE,
    cost_fn,
    label_fn,
)

representative_states = {
    "start": START_STATE,
    "blue": BLUE_STATE,
    "goal": GOAL_STATE,
}

label_cost_table = []
for name, obs in representative_states.items():
    labels = label_fn(obs)
    label_cost_table.append(
        {
            "name": name,
            "obs": int(obs),
            "labels": sorted(labels),
            "cost": float(cost_fn(labels)),
        }
    )

pprint(label_cost_table)


## Build a labelled, constrained environment

`make_env` returns a Gymnasium-style environment. The observation, reward, `terminated`, and `truncated` values stay in the usual Gymnasium positions. MASA adds labels, constraint metrics, and reward metrics to `info`.

In [ ]:
from masa.plugins.helpers import load_plugins
from masa.common.utils import make_env

load_plugins()

def build_colour_env(max_episode_steps=20, budget=0.0):
    return make_env(
        "colour_grid_world",
        "cmdp",
        max_episode_steps,
        label_fn=label_fn,
        cost_fn=cost_fn,
        budget=budget,
    )

env = build_colour_env()
obs, info = env.reset(seed=0)

print("reset obs:", obs)
print('info["labels"]:', info["labels"])
print('info["constraint"]:')
pprint(info["constraint"])
env.close()


## A small rollout helper

The helper below records only the fields we want to study. It stops as soon as either `terminated` or `truncated` becomes true.

In [ ]:
def run_rollout(actions, *, seed, max_episode_steps=20, budget=0.0):
    env = build_colour_env(max_episode_steps=max_episode_steps, budget=budget)
    obs, info = env.reset(seed=seed)
    rows = [
        {
            "event": "reset",
            "obs": int(obs),
            "labels": sorted(info["labels"]),
            "constraint_step": info["constraint"]["step"],
        }
    ]

    for step, action in enumerate(actions, start=1):
        obs, reward, terminated, truncated, info = env.step(action)
        row = {
            "event": f"step_{step}",
            "action": ACTION_NAMES[action],
            "obs": int(obs),
            "reward": float(reward),
            "terminated": bool(terminated),
            "truncated": bool(truncated),
            "labels": sorted(info["labels"]),
            "constraint_step": info["constraint"]["step"],
            "constraint_episode": info["constraint"].get("episode"),
            "metric_step": info["metrics"]["step"],
            "metric_episode": info.get("metrics", {}).get("episode"),
        }
        rows.append(row)
        if terminated or truncated:
            break

    env.close()
    return rows


## Cost rollout: reaching a labelled unsafe state

With seed `1`, four `down` actions reach the blue state. In this environment, `blue` has cost `1.0`, so the CMDP monitor reports a step violation and cumulative cost.

In [ ]:
cost_rows = run_rollout([2, 2, 2, 2], seed=1, max_episode_steps=20, budget=0.0)
pprint(cost_rows)

assert cost_rows[-1]["labels"] == ["blue"]
assert cost_rows[-1]["constraint_step"]["cost"] == 1.0
assert cost_rows[-1]["constraint_step"]["violation"] == 1.0
assert cost_rows[-1]["constraint_step"]["cum_cost"] == 1.0


## Termination rollout: reaching the goal

`terminated` means the environment task ended. With seed `4`, this scripted path reaches the goal state and receives reward `1.0`.

In [ ]:
termination_actions = [2] * 8 + [1] * 8
termination_rows = run_rollout(termination_actions, seed=4, max_episode_steps=40, budget=0.0)
pprint(termination_rows)

final_termination = termination_rows[-1]
assert final_termination["labels"] == ["goal"]
assert final_termination["reward"] == 1.0
assert final_termination["terminated"] is True
assert final_termination["truncated"] is False
assert final_termination["metric_episode"]["ep_reward"] == 1.0


## Truncation rollout: hitting the time limit

`truncated` means an external limit stopped the episode. Here the environment has `max_episode_steps=3`, so it truncates before reaching a terminal state.

In [ ]:
truncation_rows = run_rollout([1, 1, 1, 1], seed=0, max_episode_steps=3, budget=0.0)
pprint(truncation_rows)

final_truncation = truncation_rows[-1]
assert final_truncation["terminated"] is False
assert final_truncation["truncated"] is True
assert final_truncation["metric_episode"]["ep_length"] == 3


## What to remember

- `obs` and `reward` remain the environment's task interface.
- `info["labels"]` is the semantic bridge from observations to safety logic.
- `cost_fn(info["labels"])` is what cost-based constraints consume.
- `info["constraint"]["step"]` is the per-step safety view.
- `info["constraint"]["episode"]` and `info["metrics"]["episode"]` summarize the episode.
- `terminated` is task completion/failure from the environment; `truncated` is an external cutoff such as a time limit.